# Part 3 — Machine Learning: predict no-shows
Goal: a risk score the front desk can use the evening before an appointment to decide whom to phone.
Only about 1 booking in 5 is a no-show, so models are judged by recall, precision and ROC-AUC, not accuracy alone.
**Rule:** the test set is used only for reporting — never to choose a model, a setting or a threshold.

## Task 3.1 — Features, split and baseline
Features = what the front desk knows the evening before. Not used: IDs (no meaning), dates, clinic_id, gender,
and status (status IS the answer — using it would be cheating).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load Part 1's output. Do NOT shuffle or re-sort: the split depends on this row order
df = pd.read_csv("appointments_clean.csv")
print("Shape:", df.shape)

Shape: (990, 16)


In [3]:
# Features known the evening before the appointment
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()

# Yes/No -> 1/0, because models only understand numbers
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encoding: each category becomes its own 0/1 column
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

# Target: 1 = no-show, 0 = attended
y = df["no_show"]

print("X has", X.shape[1], "columns:")
print(list(X.columns))

X has 11 columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [4]:
# 80% to learn from (train), 20% kept aside to test. random_state=42 = same split every run
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("No-shows in test set:", y_test.sum())
print(f"No-show rate overall: {y.mean()*100:.1f}%")

# Baseline: a lazy 'model' that always says "Attended" (0)
baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always 'Attended'): {baseline:.4f}  ({(y_test == 0).sum()} of {len(y_test)})")

Training rows: 792
Test rows: 198
No-shows in test set: 49
No-show rate overall: 22.0%
Baseline accuracy (always 'Attended'): 0.7525  (149 of 198)


**Result:** X has 11 columns. Train 792 rows, test 198 rows, 49 no-shows in the test set (22.0% no-shows overall).
Baseline: always predicting "Attended" is already 75.25% accurate (149 of 198) while catching zero no-shows.
So any real model must beat 0.7525 accuracy AND actually catch no-shows (recall).

## Task 3.2 — Logistic regression
Scale the features (fit on train only), train a logistic regression, evaluate on the test set, and read the coefficients.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

# Scaling: every feature gets mean 0 and std 1, so big numbers (fee ~600) don't drown out small ones (reminder 0/1)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # learn mean/std from TRAIN only, then scale
X_test_s = scaler.transform(X_test)         # reuse the TRAIN mean/std on test (no peeking at test)

model = LogisticRegression(max_iter=1000)   # max_iter = enough steps for the model to finish learning
model.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000)

In [6]:
y_pred = model.predict(X_test_s)               # 0/1 decision (default cut-off 0.5)
y_prob = model.predict_proba(X_test_s)[:, 1]   # probability of no-show (column 1 = class "1")

print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall   : {recall_score(y_test, y_pred):.2f}")
print(f"F1       : {f1_score(y_test, y_pred):.2f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")   # AUC needs probabilities, not 0/1

print("\nConfusion matrix  [[TN, FP],")
print("                   [FN, TP]]")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.788
Precision: 0.63
Recall   : 0.35
F1       : 0.45
ROC-AUC  : 0.789

Confusion matrix  [[TN, FP],
                   [FN, TP]]
[[139  10]
 [ 32  17]]


In [7]:
# One coefficient per feature, in the same order as X's columns
coefs = pd.Series(model.coef_[0], index=X.columns)
print(coefs.round(3))

# reminder_sent: two ways to read its coefficient
i = list(X.columns).index("reminder_sent")    # position of reminder_sent in X
coef = model.coef_[0][i]
sd = scaler.scale_[i]                          # std of reminder_sent in the training set

print(f"\nreminder_sent coefficient (scaled): {coef:.3f}")
print(f"np.exp(coef)      = {np.exp(coef):.2f}  -> odds multiplied by this per 1 standard deviation ({sd:.2f})")
print(f"np.exp(coef / sd) = {np.exp(coef / sd):.2f}  -> odds multiplied by this for No (0) -> Yes (1)")

age                       -0.254
distance_km                0.253
fee_inr                    0.453
lead_days                  0.614
prior_no_shows             0.650
reminder_sent             -0.581
booking_channel_App       -0.047
booking_channel_Phone      0.216
booking_channel_Walk-In   -0.186
visit_type_Consultation    0.016
visit_type_Follow-up      -0.016
dtype: float64

reminder_sent coefficient (scaled): -0.581
np.exp(coef)      = 0.56  -> odds multiplied by this per 1 standard deviation (0.49)
np.exp(coef / sd) = 0.31  -> odds multiplied by this for No (0) -> Yes (1)


**Result (test set):** accuracy 0.788, precision 0.63, recall 0.35, F1 0.45, ROC-AUC 0.789.
Confusion matrix [[139, 10], [32, 17]]: the model caught 17 of the 49 no-shows and missed 32.

**Coefficients:** reminder_sent is the most negative (−0.58); lead_days (+0.61) and prior_no_shows (+0.65) are clearly positive.

**reminder_sent, two numbers:** np.exp(coef) = 0.56 is the effect of one *standard deviation* (0.49) of the scaled feature;
np.exp(coef / sd) = 0.31 is the effect of the real No → Yes change. They differ because the model was trained on scaled
features, where one unit means one standard deviation, not the 0 → 1 step.

**Conclusion:** in this model, holding the other features constant, appointments with a reminder have about 69% lower odds
of a no-show. This is an association from observational data — it does not prove that reminders cause fewer no-shows.

## Task 3.3 — Decision tree and overfitting
Compare a tree with no depth limit against a tree limited to 3 levels, on training vs test accuracy.

In [8]:
from sklearn.tree import DecisionTreeClassifier

# Trees split on cut-offs (e.g. "lead_days > 14?"), so they don't need scaled data -> use X_train, not X_train_s
tree_full = DecisionTreeClassifier(random_state=42)              # no depth limit: keeps splitting until every leaf is pure
tree_full.fit(X_train, y_train)

tree3 = DecisionTreeClassifier(random_state=42, max_depth=3)     # at most 3 levels of questions
tree3.fit(X_train, y_train)

# .score() = accuracy
print("Full-depth tree")
print("  depth:", tree_full.get_depth())
print(f"  train accuracy: {tree_full.score(X_train, y_train):.2f}")
print(f"  test accuracy : {tree_full.score(X_test, y_test):.2f}")

print("\nDepth-3 tree")
print("  depth:", tree3.get_depth())
print(f"  train accuracy: {tree3.score(X_train, y_train):.2f}")
print(f"  test accuracy : {tree3.score(X_test, y_test):.2f}")

print(f"\nBaseline accuracy (always 'Attended'): {baseline:.4f}")

Full-depth tree
  depth: 15
  train accuracy: 1.00
  test accuracy : 0.67

Depth-3 tree
  depth: 3
  train accuracy: 0.82
  test accuracy : 0.78

Baseline accuracy (always 'Attended'): 0.7525


**Result:** Full-depth tree (depth 15): train 1.00, test 0.67. Depth-3 tree: train 0.82, test 0.78.

The full tree's large gap (1.00 vs 0.67) shows overfitting: it memorised the training rows, noise included, and on new
data it does worse than the 0.7525 baseline; limiting the depth to 3 shrinks the gap and gives a model that generalises better.

## Task 3.4 — Random forest and a comparison table
A random forest = 100 decision trees, each trained on a random sample of rows and features; they vote together.

In [9]:
from sklearn.ensemble import RandomForestClassifier

# 100 trees voting together (no scaling needed, same reason as single trees)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)                 # 0/1 decision
rf_prob = rf.predict_proba(X_test)[:, 1]     # probability of no-show, for AUC

print(f"Train accuracy: {rf.score(X_train, y_train):.2f}")
print(f"Test accuracy : {accuracy_score(y_test, rf_pred):.3f}")
print(f"Recall        : {recall_score(y_test, rf_pred):.2f}")
print(f"ROC-AUC       : {roc_auc_score(y_test, rf_prob):.3f}")

# How much each feature helped the trees' splits (all values add up to 1)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 features:")
print(importances.head(3).round(3))

Train accuracy: 1.00
Test accuracy : 0.747
Recall        : 0.31
ROC-AUC       : 0.706

Top 3 features:
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64


In [10]:
# Depth-3 tree predictions (the logistic regression ones, y_pred and y_prob, come from Cell 9)
tree3_pred = tree3.predict(X_test)
tree3_prob = tree3.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "model":    ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, y_pred), accuracy_score(y_test, tree3_pred), accuracy_score(y_test, rf_pred)],
    "recall":   [recall_score(y_test, y_pred),   recall_score(y_test, tree3_pred),   recall_score(y_test, rf_pred)],
    "roc_auc":  [roc_auc_score(y_test, y_prob),  roc_auc_score(y_test, tree3_prob),  roc_auc_score(y_test, rf_prob)],
})
print(results.round(3))

                     model  accuracy  recall  roc_auc
0      Logistic regression     0.788   0.347    0.789
1  Decision tree (depth 3)     0.778   0.224    0.700
2            Random forest     0.747   0.306    0.706


**Result:** Random forest: train accuracy 1.00, test accuracy 0.747, recall 0.31, ROC-AUC 0.706.
Top 3 importances: lead_days 0.220, age 0.197, distance_km 0.195.

| model | accuracy | recall | roc_auc |
|---|---|---|---|
| Logistic regression | 0.788 | 0.347 | 0.789 |
| Decision tree (depth 3) | 0.778 | 0.224 | 0.700 |
| Random forest | 0.747 | 0.306 | 0.706 |

Logistic regression looks best: it has the highest accuracy, recall and ROC-AUC. The forest scores 1.00 on training data
but only 0.747 on the test set, so it overfits. This table is for reporting only — the formal model choice is made in
Task 3.5 using the training data alone.

## Task 3.5 — Tune, choose the model and the threshold, then test once
Step 1: choose the model using cross-validation on the training set only. The test set is not touched.

In [11]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) Try 6 forest settings (2 x 3). Each one is scored by 5-fold cross-validated ROC-AUC on the TRAINING set
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)      # takes about 30 seconds

print("Best settings:", grid.best_params_)
print(f"Best cross-validated ROC-AUC: {grid.best_score_:.3f}")

Best settings: {'max_depth': 3, 'n_estimators': 200}
Best cross-validated ROC-AUC: 0.767


In [12]:
# (b) Cross-validated ROC-AUC on the TRAINING set for each model (test set not touched)

# Logistic regression inside the pipeline supplied by the brief: the scaler is re-fitted inside every fold
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()    # UNSCALED X_train

cv_tree3 = cross_val_score(DecisionTreeClassifier(random_state=42, max_depth=3),
                           X_train, y_train, cv=5, scoring="roc_auc").mean()

cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                        X_train, y_train, cv=5, scoring="roc_auc").mean()

cv_results = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest", "Tuned random forest"],
    "cv_roc_auc": [cv_lr, cv_tree3, cv_rf, grid.best_score_],
})
print(cv_results.sort_values("cv_roc_auc", ascending=False).round(3))

                     model  cv_roc_auc
0      Logistic regression       0.774
3      Tuned random forest       0.767
2            Random forest       0.747
1  Decision tree (depth 3)       0.678


**(a)** Best forest settings: {'max_depth': 3, 'n_estimators': 200}, cross-validated ROC-AUC 0.767.

**(b)** Cross-validated ROC-AUC on the training set: logistic regression 0.774, tuned forest 0.767, random forest 0.747,
depth-3 tree 0.678. **Logistic regression is the highest, so it is the final model.** The choice used training data only.

Step 2: choose the threshold on training data, then evaluate the chosen model once on the test set.

In [13]:
# (c) Choose the threshold WITHOUT the test set.
# Out-of-fold: each training row gets its probability from a model that did NOT see that row during training
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for threshold in [0.5, 0.35]:
    flag = (oof >= threshold).astype(int)       # 1 = flag this booking as risky
    print(f"Threshold {threshold}: "
          f"precision {precision_score(y_train, flag):.2f}, "
          f"recall {recall_score(y_train, flag):.2f}, "
          f"F1 {f1_score(y_train, flag):.2f}, "
          f"flagged {flag.sum()}")

Threshold 0.5: precision 0.58, recall 0.25, F1 0.35, flagged 73
Threshold 0.35: precision 0.51, recall 0.47, F1 0.49, flagged 154


In [14]:
THRESHOLD = 0.35     # chosen in (c): higher F1 and much higher recall

# (d) Fit the winner on the whole training set (the pipeline fits its own scaler), then test ONCE
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]          # risk score for each test booking
test_flag = (probs >= THRESHOLD).astype(int)

print(f"Test set at threshold {THRESHOLD}:")
print(f"  precision: {precision_score(y_test, test_flag):.2f}")
print(f"  recall   : {recall_score(y_test, test_flag):.2f}")
print(f"  flagged  : {test_flag.sum()} of {len(y_test)}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, probs):.3f}")

Test set at threshold 0.35:
  precision: 0.57
  recall   : 0.57
  flagged  : 49 of 198
  ROC-AUC  : 0.789


In [15]:
# (e) One row per test booking. X_test kept the original row numbers, so we can look up appointment_id in df
risk = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"],
    "no_show": y_test,
    "risk_prob": probs,          # not rounded, so flag always agrees with risk_prob >= 0.35
    "flag": test_flag,
})
risk.to_csv("test_risk_scores.csv", index=False)

print("Saved test_risk_scores.csv:", risk.shape)
print("Rows with flag = 1:", risk["flag"].sum())
print("\nHighest-risk bookings:")
print(risk.sort_values("risk_prob", ascending=False).head())

Saved test_risk_scores.csv: (198, 4)
Rows with flag = 1: 49

Highest-risk bookings:
    appointment_id  no_show  risk_prob  flag
626          A0635        1   0.937204     1
536          A0544        1   0.924066     1
289          A0293        1   0.919634     1
939          A0950        1   0.818616     1
985          A0996        0   0.766710     1


**(c) Threshold, chosen on out-of-fold training probabilities:**
- 0.5 → precision 0.58, recall 0.25, F1 0.35, 73 flagged
- 0.35 → precision 0.51, recall 0.47, F1 0.49, 154 flagged

**0.35 is chosen:** higher F1 and much higher recall. Recall matters here because a missed warning costs an empty doctor
slot, while an extra call only costs the front desk about a minute.

**(d) One test-set evaluation** (logistic regression, threshold 0.35): precision 0.57, recall 0.57, 49 of 198 flagged, ROC-AUC 0.789.

**(e)** test_risk_scores.csv: 198 rows (appointment_id, no_show, risk_prob, flag), 49 flagged.
In practice, a clinic would set the threshold by how many follow-up calls the front desk can make in an evening,
weighed against the cost of an empty slot.